# MINI Cells — Experiment 009: 2D Latent Tissue

Primary comparison: unchanged `minicells-v2` versus a causal K=4 latent-tissue NCA. Both train from random initialization for 2M consumed TinyStories tokens. The 2D model keeps the Experiment 006 language recipe and adds only a latent tissue axis with local vertical communication.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

ROOT = Path('/kaggle/working/mini-cells')
REF = os.environ.get('MINICELLS_REF', 'main')
os.chdir('/kaggle/working')
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REF, 'https://github.com/ArcheLabs/mini-cells.git', str(ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm,dev]'], cwd=ROOT, check=True)
os.chdir(ROOT)
print('repo:', ROOT)
print('ref:', REF)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
if not torch.cuda.is_available():
    raise RuntimeError('Experiment 009 requires CUDA')


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', 'tests/research/01-foundations/test_language_2d.py', 'tests/research/01-foundations/test_language_scaling.py', '-q'], cwd=ROOT, check=True)


## Run the primary 1D vs K=4 comparison

With two T4 GPUs the two models run concurrently, one process per GPU. On a single GPU they run sequentially. Add `--include-k2` only after the primary comparison if a K=2 follow-up is useful.


In [ ]:
subprocess.run([sys.executable, 'scripts/research/run_language_2d.py'], cwd=ROOT, check=True)


In [ ]:
import json
import pandas as pd
from IPython.display import Image, Markdown, display

OUT = ROOT / 'results' / 'language-2d-latent-tissue-v1'
decision = json.loads((OUT / 'decision.json').read_text(encoding='utf-8'))
summary = pd.read_csv(OUT / 'model-summary.csv')
display(Markdown(f"## {decision['status']}: {decision['diagnosis']}"))
display(summary)
display(Markdown(
    f"**2D/1D PPL @2M:** {decision['comparison']['ppl_ratio_2d_to_1d_at_2m']:.4f}×  \
"
    f"**Parameter ratio:** {decision['comparison']['parameter_ratio_2d_to_1d']:.4f}×  \
"
    f"**Throughput ratio:** {decision['comparison']['throughput_ratio_2d_to_1d']:.4f}×"
))
for name in ['ppl-comparison.png', 'throughput.png', 'tissue-cosine.png']:
    display(Image(filename=str(OUT / name)))
display(Markdown('### Tissue diagnostics'))
print(json.dumps(decision['tissue'], indent=2))


## Publish curated results

Add a Kaggle Secret named `GITHUB_TOKEN` containing a fine-grained GitHub token with **Contents: Read and write**. After reviewing the outputs above, set `PUBLISH=True`. The publisher commits only curated Experiment 009 artifacts to `kaggle/experiment-009-results`; it does not push caches or logs.


In [ ]:
# Set this to True only after reviewing decision.json and the plots.
PUBLISH = False
if PUBLISH:
    subprocess.run([sys.executable, 'scripts/research/publish_experiment_009_results.py', '--push'], cwd=ROOT, check=True)
